In [0]:
%pip install xgboost prophet python-dateutil

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

N_CUSTOMERS  = 10_000
START_DATE   = date(2022, 1, 1)
N_MONTHS     = 36
SEGMENTS     = ['Enterprise', 'Mid-Market', 'SMB']
SEG_DIST     = [0.10, 0.25, 0.65]
PLANS        = ['Basic', 'Pro', 'Enterprise']
PLAN_MRR     = {'Basic': 99, 'Pro': 299, 'Enterprise': 999}
ANNUAL_CHURN = {'Enterprise': 0.05, 'Mid-Market': 0.12, 'SMB': 0.22}

months = [START_DATE + relativedelta(months=m) for m in range(N_MONTHS)]
print("✓ Config ready")

✓ Config ready


In [0]:
segments = np.random.choice(SEGMENTS, N_CUSTOMERS, p=SEG_DIST)

def assign_plan(seg):
    if seg == 'Enterprise':   return np.random.choice(PLANS, p=[0.05, 0.25, 0.70])
    elif seg == 'Mid-Market': return np.random.choice(PLANS, p=[0.20, 0.55, 0.25])
    else:                     return np.random.choice(PLANS, p=[0.60, 0.35, 0.05])

plans        = [assign_plan(s) for s in segments]
signup_dates = [START_DATE + relativedelta(months=int(m))
                for m in np.random.randint(0, N_MONTHS, N_CUSTOMERS)]

customers_pd = pd.DataFrame({
    'customer_id': range(1, N_CUSTOMERS + 1),
    'signup_date': [str(d) for d in signup_dates],
    'segment':     segments,
    'plan_type':   plans,
    'base_mrr':    [float(PLAN_MRR[p]) for p in plans],
})

def assign_churn_date(row):
    monthly_churn = 1 - (1 - ANNUAL_CHURN[row['segment']]) ** (1/12)
    for m in range(N_MONTHS):
        month = START_DATE + relativedelta(months=m)
        if month < date.fromisoformat(row['signup_date']): continue
        if np.random.random() < monthly_churn: return str(month)
    return None

customers_pd['churn_date'] = customers_pd.apply(assign_churn_date, axis=1)

print(f"Customers: {len(customers_pd):,}")
print(customers_pd['segment'].value_counts().to_string())

mrr_rows = []
for _, c in customers_pd.iterrows():
    signup = date.fromisoformat(c['signup_date'])
    churn  = date.fromisoformat(c['churn_date']) if c['churn_date'] else None
    for month in months:
        if month < signup: continue
        if churn and month > churn: continue
        tenure = (month.year - signup.year)*12 + (month.month - signup.month)
        drift  = max(0.7, min(1 + 0.002*tenure + np.random.normal(0, 0.01), 1.5))
        mrr_rows.append({'customer_id': int(c['customer_id']), 'month': str(month),
                         'mrr': round(c['base_mrr'] * drift, 2)})

mrr_pd = pd.DataFrame(mrr_rows)
print(f"\nMRR rows: {len(mrr_pd):,}")

activity_rows = []
for _, c in customers_pd.iterrows():
    signup = date.fromisoformat(c['signup_date'])
    churn  = date.fromisoformat(c['churn_date']) if c['churn_date'] else None
    base_logins  = {'Enterprise': 45, 'Mid-Market': 25, 'SMB': 12}[c['segment']]
    base_tickets = {'Enterprise': 1.5, 'Mid-Market': 1.0, 'SMB': 0.5}[c['segment']]
    for month in months:
        if month < signup: continue
        if churn and month > churn: continue
        mtc = (churn.year - month.year)*12 + (churn.month - month.month) if churn else 999
        login_mult  = max(0.1, 1 - 0.25*(3 - mtc)) if mtc <= 3 else 1.0
        ticket_mult = 2.5 if mtc <= 2 else (1.5 if mtc <= 4 else 1.0)
        activity_rows.append({
            'customer_id':     int(c['customer_id']),
            'month':           str(month),
            'logins':          max(0, int(np.random.poisson(base_logins * login_mult))),
            'support_tickets': max(0, int(np.random.poisson(base_tickets * ticket_mult))),
        })

activity_pd = pd.DataFrame(activity_rows)
print(f"Activity rows: {len(activity_pd):,}")

Customers: 10,000
segment
SMB           6445
Mid-Market    2524
Enterprise    1031

MRR rows: 154,551
Activity rows: 154,551


In [0]:
customers_spark  = spark.createDataFrame(customers_pd)
mrr_spark        = spark.createDataFrame(mrr_pd)
activity_spark   = spark.createDataFrame(activity_pd)

customers_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_customers")
mrr_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_mrr")
activity_spark.write.format("delta").mode("overwrite").saveAsTable("bronze_activity")

print("Bronze layer written")
print(f"bronze_customers: {spark.table('bronze_customers').count():,} rows")
print(f"bronze_mrr: {spark.table('bronze_mrr').count():,} rows")
print(f"bronze_activity: {spark.table('bronze_activity').count():,} rows")

Bronze layer written
bronze_customers: 10,000 rows
bronze_mrr: 154,551 rows
bronze_activity: 154,551 rows


In [0]:
silver_customers = spark.sql("""
    SELECT customer_id, CAST(signup_date AS DATE)  AS signup_date,
    CAST(churn_date  AS DATE)  AS churn_date, segment, plan_type, base_mrr
    FROM bronze_customers
    WHERE customer_id IS NOT NULL""")

silver_mrr = spark.sql("""SELECT customer_id,
        CAST(month AS DATE) AS month,
        CAST(mrr AS DOUBLE) AS mrr
    FROM bronze_mrr
    WHERE mrr > 0""")

silver_activity = spark.sql("""
    SELECT customer_id, CAST(month AS DATE) AS month,
    CAST(logins AS INT) AS logins,
    CAST(support_tickets AS INT) AS support_tickets
    FROM bronze_activity
    WHERE logins >= 0
      AND support_tickets >= 0 """)

silver_customers.write.format("delta").mode("overwrite").saveAsTable("silver_customers")
silver_mrr.write.format("delta").mode("overwrite").saveAsTable("silver_mrr")
silver_activity.write.format("delta").mode("overwrite").saveAsTable("silver_activity")

print("Silver layer written")
print(f"silver_customers: {silver_customers.count():,} rows")
print(f"silver_mr: {silver_mrr.count():,} rows")
print(f"silver_activity: {silver_activity.count():,} rows")

Silver layer written
silver_customers: 10,000 rows
silver_mr: 154,551 rows
silver_activity: 154,551 rows


In [0]:
gold_features = spark.sql("""
    SELECT
        m.customer_id,
        m.month,
        m.mrr,
        m.mrr * 12                                          AS arr,
        a.logins,
        a.support_tickets,
        c.segment,
        c.plan_type,
        c.signup_date,
        c.churn_date,

        FLOOR(DATEDIFF(m.month, c.signup_date) / 30.44)    AS tenure_months,

        AVG(a.logins) OVER (
            PARTITION BY m.customer_id
            ORDER BY m.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )                                                   AS rolling_3m_logins,

        AVG(a.logins) OVER (
            PARTITION BY m.customer_id
            ORDER BY m.month
            ROWS BETWEEN 5 PRECEDING AND CURRENT ROW
        )                                                   AS rolling_6m_logins,

        AVG(a.support_tickets) OVER (
            PARTITION BY m.customer_id
            ORDER BY m.month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )                                                   AS rolling_ticket_3m,

        (m.mrr - LAG(m.mrr) OVER (
            PARTITION BY m.customer_id ORDER BY m.month)
        ) / NULLIF(LAG(m.mrr) OVER (
            PARTITION BY m.customer_id ORDER BY m.month), 0) AS mrr_growth_rate,

        CASE
            WHEN c.churn_date IS NOT NULL
             AND c.churn_date <= ADD_MONTHS(m.month, 1) THEN 1
            ELSE 0
        END                                                 AS churn_next_month,

        CASE c.segment
            WHEN 'SMB'        THEN 0
            WHEN 'Mid-Market' THEN 1
            WHEN 'Enterprise' THEN 2
        END                                                 AS segment_enc,

        CASE c.plan_type
            WHEN 'Basic'      THEN 0
            WHEN 'Pro'        THEN 1
            WHEN 'Enterprise' THEN 2
        END                                                 AS plan_enc

    FROM silver_mrr       m
    JOIN silver_activity  a ON m.customer_id = a.customer_id AND m.month = a.month
    JOIN silver_customers c ON m.customer_id = c.customer_id""")

gold_features = gold_features.fillna({'mrr_growth_rate': 0.0})

gold_features.write.format("delta").mode("overwrite").saveAsTable("gold_customer_features")

print("Gold layer written")
print(f"gold_customer_features: {gold_features.count():,} rows")
print(f"Churn label rate: {gold_features.filter('churn_next_month=1').count() / gold_features.count():.2%}")

Gold layer written
gold_customer_features: 154,551 rows
Churn label rate: 3.08%


In [0]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

FEATURES = ['mrr','arr','mrr_growth_rate','tenure_months',
            'logins','rolling_3m_logins','rolling_6m_logins',
            'support_tickets','rolling_ticket_3m','segment_enc','plan_enc']

feat_pd = spark.table("gold_customer_features").toPandas()
feat_pd['tenure_months'] = pd.to_numeric(feat_pd['tenure_months'])
feat_pd = feat_pd.dropna(subset=FEATURES)
feat_pd['mrr_growth_rate'] = feat_pd['mrr_growth_rate'].clip(-0.5, 0.5)

feat_pd['month'] = pd.to_datetime(feat_pd['month'])
cutoff   = sorted(feat_pd['month'].unique())[-3]
train_df = feat_pd[feat_pd['month'] < cutoff]
test_df  = feat_pd[feat_pd['month'] >= cutoff]

X_train, y_train = train_df[FEATURES], train_df['churn_next_month']
X_test,  y_test  = test_df[FEATURES],  test_df['churn_next_month']

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    scale_pos_weight=scale_pos, eval_metric='auc',
    random_state=42, verbosity=0
)
model.fit(X_train, y_train,
          eval_set=[(X_test, y_test)], verbose=False)

auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print(f"✓ Model trained  |  Test AUC: {auc:.4f}")

fi = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 5 features:")
print(fi.head(5).to_string())

✓ Model trained  |  Test AUC: 0.9855

Top 5 features:
logins               0.606347
rolling_ticket_3m    0.137810
segment_enc          0.135013
rolling_3m_logins    0.043087
support_tickets      0.018003


In [0]:
feat_pd['churn_probability'] = model.predict_proba(feat_pd[FEATURES])[:, 1]
feat_pd['revenue_risk']      = feat_pd['churn_probability'] * feat_pd['arr']

latest_month = feat_pd['month'].max()
latest_pd    = feat_pd[feat_pd['month'] == latest_month].copy()

total_arr  = latest_pd['arr'].sum()
total_risk = latest_pd['revenue_risk'].sum()
print(f"Total ARR:       ${total_arr:>12,.0f}")
print(f"ARR at risk:     ${total_risk:>12,.0f}  ({total_risk/total_arr:.1%})")
print(f"Avg churn prob:  {latest_pd['churn_probability'].mean():.2%}")

print("\nRevenue risk by segment:")
seg = latest_pd.groupby('segment').agg(
    customers=('customer_id','count'),
    total_arr=('arr','sum'),
    arr_at_risk=('revenue_risk','sum')
).assign(pct_at_risk=lambda d: d['arr_at_risk']/d['total_arr'])
print(seg.sort_values('arr_at_risk', ascending=False).to_string())

risk_spark = spark.createDataFrame(
    latest_pd[['customer_id','segment','plan_type','arr','churn_probability','revenue_risk']]
)
risk_spark.write.format("delta").mode("overwrite").saveAsTable("gold_revenue_risk")

monthly_arr = feat_pd.groupby('month')['mrr'].sum().reset_index()
monthly_arr.columns = ['ds', 'y']
monthly_arr['y'] = monthly_arr['y'] * 12
spark.createDataFrame(monthly_arr.astype(str)).write.format("delta").mode("overwrite").saveAsTable("gold_monthly_arr")

Total ARR:       $  33,216,481
ARR at risk:     $   2,019,693  (6.1%)
Avg churn prob:  8.66%

Revenue risk by segment:
            customers    total_arr   arr_at_risk  pct_at_risk
segment                                                      
SMB              4581  12152175.96  1.639483e+06     0.134913
Mid-Market       2119  11795542.92  3.340661e+05     0.028321
Enterprise        957   9268762.56  4.614408e+04     0.004978


In [0]:
%pip install prophet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from prophet import Prophet
import pandas as pd

monthly_arr = spark.table("gold_monthly_arr").toPandas()
monthly_arr['ds'] = pd.to_datetime(monthly_arr['ds'])
monthly_arr['y']  = monthly_arr['y'].astype(float)
monthly_arr = monthly_arr.sort_values('ds').reset_index(drop=True)

print("Historical ARR:")
print(monthly_arr.tail(6).to_string(index=False))

model_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05   # controls trend flexibility
)
model_prophet.fit(monthly_arr)

future   = model_prophet.make_future_dataframe(periods=12, freq='MS')
forecast = model_prophet.predict(future)

forecast_clean = forecast[['ds','yhat','yhat_lower','yhat_upper']].copy()
forecast_clean.columns = ['month','arr_forecast','arr_lower','arr_upper']

print(f"\n12-month ARR forecast:")
print(forecast_clean.tail(12).to_string(index=False))

Historical ARR:
        ds           y
2024-07-01 29812301.76
2024-08-01 30575931.84
2024-09-01 31282548.96
2024-10-01 31842675.84
2024-11-01 32565745.20
2024-12-01 33216481.44


22:26:01 - cmdstanpy - INFO - Chain [1] start processing
22:26:15 - cmdstanpy - INFO - Chain [1] done processing



12-month ARR forecast:
     month  arr_forecast    arr_lower    arr_upper
2025-01-01  3.466234e+07 3.458990e+07 3.473442e+07
2025-02-01  3.564210e+07 3.541008e+07 3.589006e+07
2025-03-01  3.626232e+07 3.580444e+07 3.674219e+07
2025-04-01  3.728658e+07 3.658420e+07 3.805051e+07
2025-05-01  3.808493e+07 3.708924e+07 3.918318e+07
2025-06-01  3.886994e+07 3.750862e+07 4.027120e+07
2025-07-01  3.967225e+07 3.799764e+07 4.143038e+07
2025-08-01  4.034292e+07 3.826767e+07 4.247726e+07
2025-09-01  4.105135e+07 3.858327e+07 4.358644e+07
2025-10-01  4.163123e+07 3.870863e+07 4.459943e+07
2025-11-01  4.244160e+07 3.908806e+07 4.581167e+07
2025-12-01  4.316661e+07 3.937737e+07 4.703236e+07


In [0]:
last_historical = monthly_arr['ds'].max()

forecast_clean['type'] = forecast_clean['month'].apply(
    lambda d: 'historical' if d <= last_historical else 'forecast'
)

spark.createDataFrame(forecast_clean.astype(str)) \
     .write.format("delta").mode("overwrite") \
     .saveAsTable("gold_arr_forecast")

print(f"  Historical months : {(forecast_clean['type']=='historical').sum()}")
print(f"  Forecast months   : {(forecast_clean['type']=='forecast').sum()}")

  Historical months : 36
  Forecast months   : 12


In [0]:
risk_pd = spark.table("gold_revenue_risk").toPandas()

# Base: current churn probabilities
# Bear: churn 10% worse (multiply probabilities up, cap at 1.0)
# Bull: churn 10% better (multiply probabilities down)

scenarios = {}
for name, churn_mult in [('Bear', 1.10), ('Base', 1.00), ('Bull', 0.90)]:
    s = risk_pd.copy()
    s['churn_probability'] = (s['churn_probability'] * churn_mult).clip(upper=1.0)
    s['revenue_risk']      = s['churn_probability'] * s['arr']
    s['scenario']          = name
    scenarios[name]        = s

scenario_pd = pd.concat(scenarios.values(), ignore_index=True)

print(f"{'Scenario':<10} {'Total ARR':>14} {'ARR at Risk':>14} {'% at Risk':>10}")
for name in ['Bear', 'Base', 'Bull']:
    s       = scenarios[name]
    arr     = s['arr'].sum()
    at_risk = s['revenue_risk'].sum()
    print(f"{name:<10} ${arr:>13,.0f} ${at_risk:>13,.0f} {at_risk/arr:>9.1%}")

forecast_future = forecast_clean[forecast_clean['type'] == 'forecast'].copy()
scenario_forecasts = []

for name, churn_mult in [('Bear', 1.10), ('Base', 1.00), ('Bull', 0.90)]:
    sf = forecast_future.copy()
    adjustment = 1 - ((churn_mult - 1) * 0.5)   
    sf['arr_forecast'] = sf['arr_forecast'].astype(float) * adjustment
    sf['arr_lower']    = sf['arr_lower'].astype(float)    * adjustment
    sf['arr_upper']    = sf['arr_upper'].astype(float)    * adjustment
    sf['scenario']     = name
    scenario_forecasts.append(sf)

scenario_forecast_pd = pd.concat(scenario_forecasts, ignore_index=True)

print("\n12-month forecast by scenario (last 3 months shown):")
print(scenario_forecast_pd.groupby('scenario').tail(1)[
    ['scenario','month','arr_forecast']
].sort_values('arr_forecast', ascending=False).to_string(index=False))

Scenario        Total ARR    ARR at Risk  % at Risk
Bear       $   33,216,481 $    2,180,611      6.6%
Base       $   33,216,481 $    2,019,693      6.1%
Bull       $   33,216,481 $    1,817,724      5.5%

12-month forecast by scenario (last 3 months shown):
scenario      month  arr_forecast
    Bull 2025-12-01  4.532494e+07
    Base 2025-12-01  4.316661e+07
    Bear 2025-12-01  4.100828e+07


In [0]:
spark.createDataFrame(
    scenario_pd[['customer_id','segment','plan_type','arr','churn_probability','revenue_risk','scenario']]
).write.format("delta").mode("overwrite").saveAsTable("gold_scenario_risk")

spark.createDataFrame(scenario_forecast_pd.astype(str)) \
     .write.format("delta").mode("overwrite") \
     .saveAsTable("gold_scenario_forecast")

In [0]:
seg_risk = risk_pd.groupby('segment').agg(
    customers    = ('customer_id', 'count'),
    total_arr    = ('arr', 'sum'),
    arr_at_risk  = ('revenue_risk', 'sum'),
    avg_churn    = ('churn_probability', 'mean')
).reset_index()
seg_risk['pct_at_risk'] = seg_risk['arr_at_risk'] / seg_risk['total_arr']
seg_risk = seg_risk.sort_values('arr_at_risk', ascending=False)

plan_risk = risk_pd.groupby('plan_type').agg(
    customers    = ('customer_id', 'count'),
    total_arr    = ('arr', 'sum'),
    arr_at_risk  = ('revenue_risk', 'sum'),
    avg_churn    = ('churn_probability', 'mean')
).reset_index()
plan_risk['pct_at_risk'] = plan_risk['arr_at_risk'] / plan_risk['total_arr']

print("Risk by segment:")
print(seg_risk.to_string(index=False))
print("\nRisk by plan:")
print(plan_risk.to_string(index=False))

spark.createDataFrame(seg_risk).write.format("delta").mode("overwrite").saveAsTable("gold_risk_by_segment")
spark.createDataFrame(plan_risk).write.format("delta").mode("overwrite").saveAsTable("gold_risk_by_plan")

Risk by segment:
   segment  customers   total_arr  arr_at_risk  avg_churn  pct_at_risk
       SMB       4581 12152175.96 1.639483e+06   0.131660     0.134913
Mid-Market       2119 11795542.92 3.340661e+05   0.026398     0.028321
Enterprise        957  9268762.56 4.614408e+04   0.004026     0.004978

Risk by plan:
 plan_type  customers   total_arr   arr_at_risk  avg_churn  pct_at_risk
     Basic       3233  3959462.88 448064.999471   0.113549     0.113163
Enterprise       1483 18374493.12 686458.569365   0.037513     0.037359
       Pro       2941 10882525.44 885169.577426   0.081670     0.081339


In [0]:
display(spark.table("gold_risk_by_segment"))

segment,customers,total_arr,arr_at_risk,avg_churn,pct_at_risk
SMB,4581,1.215217596E7,1639483.0087283538,0.13166046,0.13491271144565897
Mid-Market,2119,1.179554292E7,334066.05666266836,0.02639829,0.028321380281380752
Enterprise,957,9268762.56,46144.080869968864,0.004026008,0.004978451068442178


In [0]:
display(spark.table("gold_risk_by_plan"))

plan_type,customers,total_arr,arr_at_risk,avg_churn,pct_at_risk
Basic,3233,3959462.88,448064.9994706773,0.113549404,0.11316307616720915
Enterprise,1483,1.837449312E7,686458.5693647298,0.037513074,0.037359320057517305
Pro,2941,1.088252544E7,885169.5774255838,0.08166963,0.08133861779656762


In [0]:
display(spark.table("gold_revenue_risk"))

customer_id,segment,plan_type,arr,churn_probability,revenue_risk
1,SMB,Basic,1222.92,0.12477074,152.58463114053012
4,SMB,Pro,3751.56,7.892212E-4,2.9608106909156775
5,Mid-Market,Pro,3671.3999999999996,0.020246794,74.33408113382755
7,Enterprise,Basic,1220.8799999999999,2.3307986E-4,0.2845625391765497
8,SMB,Pro,3615.96,0.0028615748,10.347339861374348
9,SMB,Pro,3649.92,0.023627995,86.24028998494148
10,SMB,Pro,3765.84,0.0065379827,24.62099687766284
11,Enterprise,Enterprise,12637.68,6.378156E-5,0.8060509583103703
12,SMB,Pro,3747.4800000000005,0.002530341,9.48240242790431
13,SMB,Pro,3581.76,0.095219135,341.0520899963379


In [0]:
display(spark.table("gold_arr_forecast"))

month,arr_forecast,arr_lower,arr_upper,type
2022-01-01,1005931.6846984299,1005931.6846686292,1005931.684727989,historical
2022-02-01,2161805.209206159,2161805.2091782046,2161805.209236259,historical
2022-03-01,3098578.0463831825,3098578.0463555907,3098578.046415895,historical
2022-04-01,4193242.304615959,4193242.3045858894,4193242.304645963,historical
2022-05-01,5229824.2735902425,5229824.273560857,5229824.273620837,historical
2022-06-01,6090406.981487518,6090406.9814585745,6090406.981516774,historical
2022-07-01,7088583.595593558,7088583.595563685,7088583.595620751,historical
2022-08-01,8053067.024280835,8053067.024251957,8053067.024312049,historical
2022-09-01,9151879.80301036,9151879.802978994,9151879.803039324,historical
2022-10-01,10026325.272558551,10026325.272530159,10026325.272589445,historical


In [0]:
display(spark.table("gold_scenario_forecast"))

month,arr_forecast,arr_lower,arr_upper,type,scenario
2025-01-01,32929221.62353851,32860409.233871948,32997698.606775388,forecast,Bear
2025-02-01,33859994.379896715,33639574.17706153,34095553.15319822,forecast,Bear
2025-03-01,34449207.56624076,34014220.6984837,34905075.93024487,forecast,Bear
2025-04-01,35422253.80712866,34754994.26891123,36147982.213099524,forecast,Bear
2025-05-01,36180682.924714305,35234781.6086298,37224023.29426998,forecast,Bear
2025-06-01,36926446.664160654,35633189.436303124,38257642.02139406,forecast,Bear
2025-07-01,37688640.78762845,36097761.25006894,39358863.222807795,forecast,Bear
2025-08-01,38325776.84156085,36354285.467064284,40353397.38404281,forecast,Bear
2025-09-01,38998783.72015476,36654110.6639121,41407115.21405939,forecast,Bear
2025-10-01,39549665.26294248,36773193.81417045,42369455.870139055,forecast,Bear


In [0]:
display(spark.table("gold_scenario_risk"))

customer_id,segment,plan_type,arr,churn_probability,revenue_risk,scenario
1,SMB,Basic,1222.92,0.13724782,167.84309881031515,Bear
4,SMB,Pro,3751.56,8.6814334E-4,3.256891825518105,Bear
5,Mid-Market,Pro,3671.3999999999996,0.022271475,81.76749266646802,Bear
7,Enterprise,Basic,1220.8799999999999,2.5638784E-4,0.31301878243451936,Bear
8,SMB,Pro,3615.96,0.0031477322,11.382073847511784,Bear
9,SMB,Pro,3649.92,0.025990795,94.86432374238969,Bear
10,SMB,Pro,3765.84,0.007191781,27.083096390068533,Bear
11,Enterprise,Enterprise,12637.68,7.015972E-5,0.886656044946285,Bear
12,SMB,Pro,3747.4800000000005,0.002783375,10.430642496189105,Bear
13,SMB,Pro,3581.76,0.10474105,375.15730967044834,Bear


In [0]:
display(spark.table("gold_monthly_arr"))

ds,y
2022-01-01,1005931.6799999999
2022-02-01,2161805.16
2022-03-01,3098577.96
2022-04-01,4193242.1999999997
2022-05-01,5229824.16
2022-06-01,6090406.800000001
2022-07-01,7088583.36
2022-08-01,8053066.800000001
2022-09-01,9151879.56
2022-10-01,10026324.959999999
